In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

import coincidence_v4

In [ ]:
fname = r"J:\ctgroup\Edward\DATA\VMI\20251015\Xe S Calibration.cv4"

data = coincidence_v4.load_file(fname, coincidence=False)

In [ ]:
x, y, t, etof, _ = data
etof += 0.26 * np.random.random_sample(len(etof))
df = pd.DataFrame({'x': x, 'y': y, 't': t, 'etof': etof})
df = df[(df['t'] > 0) & (df['t'] < 20000) & (df['etof'] > 495) & (df['etof'] < 510)]
df = df.sample(frac=1).reset_index(drop=True)


In [ ]:
px.histogram(x=df.t, nbins=10000, title='ToA Spectra (Linear Scale, Count)', labels={'value': 'Time (ns)'},
             template='plotly_white', log_y=True).show()

px.histogram(x=df.etof, nbins=10000, title='e-ToF Spectra (Linear Scale, Count)', labels={'value': 'Time (ns)'},
             template='plotly_white').show()

# px.histogram(x=df.itof, nbins=10000, title='i-ToF Spectra (Linear Scale, Count)', labels={'value': 'Time (ns)'},
#              template='plotly_white').show()

px.density_heatmap(x=df.x, y=df.y, nbinsx=512, nbinsy=512, title='Detector Hit Map',
                   labels={'x': 'X (pixels)', 'y': 'Y (pixels)'}, template='plotly_white',
                   color_continuous_scale=px.colors.sequential.Inferno, height=800, width=800,
                   # range_color=(0,4/0)
                   ).show()


In [ ]:
cx, cy = 128, 128
c_etof = 501
angle = -0.55

In [ ]:
df_filt = df[(df['etof'] > c_etof - 10) & (df['etof'] < c_etof + 10)]

px.density_heatmap(
        df_filt,
        x="x", y="y",
        nbinsx=512, nbinsy=512,
        title="Detector Hit Map (Filtered on e-ToF around 501 ns)",
        width=800, height=800,
        color_continuous_scale=px.colors.sequential.Inferno,
        # range_color=(0,200)
)

In [ ]:
df_filt['x_rot'] = (df_filt['x'] - cx) * np.cos(-angle) - (df_filt['y'] - cy) * np.sin(-angle)
df_filt['y_rot'] = (df_filt['x'] - cx) * np.sin(-angle) + (df_filt['y'] - cy) * np.cos(-angle)
px.density_heatmap(
        df_filt,
        x="x_rot", y="etof",
        nbinsx=512, nbinsy=512,
        title="Detector Hit Map (Filtered on e-ToF around 501 ns, Rotated)",
        width=800, height=800,
        color_continuous_scale=px.colors.sequential.Inferno,
        # range_color=(0,200)
)

In [ ]:
import plotly.graph_objects as go

h3d = np.histogramdd(
        df_filt[['x_rot', 'y_rot', 'etof']].to_numpy(),
        bins=(128, 128, 128),
        range=((-128, 128), (-128, 128), (490, 510))
)[0]

go.Figure(
        data=go.Volume(
                x=np.repeat(np.linspace(-128, 128, 512), 512 * 512),
                y=np.tile(np.repeat(np.linspace(-128, 128, 512), 512), 512),
                z=np.tile(np.linspace(490, 510, 512), 512 * 512),
                value=h3d.flatten(),
                isomin=1,
                isomax=h3d.max(),
                opacity=0.1,
                surface_count=20,
                colorscale='Inferno',
        )
).write_html("volume_plot.html")